In [11]:
import numpy as np
import json
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
from datetime import datetime

In [12]:
num_perm = 1000 
model_numbers = ["model_1", "model_2", "model_1a", "model_2a", "model_1b", "model_2b"]
model_types = ["xgboost", "rf"]
model_combos = [(mn, mt) for mn in model_numbers for mt in model_types]

In [13]:
for model_number, model_type in model_combos:
    combo_start_time = datetime.now()
    print(f"Processing {model_number} | Type: {model_type}")
    
    json_path = f'results/{model_number}_{model_type}_best_within_one.json'
    with open(json_path, 'r') as f:
            params = json.load(f)    

    # param clean
    for key in ["number", "value", "std_err", "achieved_value"]:
        params.pop(key, None)

    params["n_estimators"] = 250
    for k in ["max_depth", "min_samples_leaf", "min_samples_split", "min_child_weight"]:
        if k in params:
            try:
                params[k] = int(params[k])
            except:
                pass

    # load data   
    x_path = f"data/X_train_{model_number}.csv"
    y_path = f"data/y_train_{model_number}.csv"
        
        
    X_train_raw = pd.read_csv(x_path, keep_default_na=False)
    y_train_raw = pd.read_csv(y_path, keep_default_na=False).values.ravel()

    # permute feature importance
    ans = []
    for i in range(num_perm):
        # upsample
        upsampler = RandomOverSampler(random_state=i)
        X_res, y_res = upsampler.fit_resample(X_train_raw, y_train_raw)
            
        # model initiate
        if model_type == "xgboost":
            model = XGBClassifier(**params, n_jobs=-1, random_state=i, verbosity=0)
        else:
            model = RandomForestClassifier(**params, n_jobs=-1, random_state=i)
            
        # train the model
        model.fit(X_res, y_res)
            
        # get result
        importance_dict = dict(zip(X_train_raw.columns, model.feature_importances_))
        ans.append(importance_dict)
            
        # check process
        if (i + 1) % 100 == 0:
            print(f"Process: {i + 1}/{num_perm}")

        # save result
    df = pd.DataFrame(ans)
    output_name = f"results/{model_number}_{model_type}_feature_importance.csv"
    df.to_csv(output_name, index=False)
        
    combo_time = datetime.now() - combo_start_time
    print(f"Save: {output_name} | Time: {combo_time}")

Processing model_1 | Type: xgboost
Process: 100/1000
Process: 200/1000
Process: 300/1000
Process: 400/1000
Process: 500/1000
Process: 600/1000
Process: 700/1000
Process: 800/1000
Process: 900/1000
Process: 1000/1000
Save: results/model_1_xgboost_feature_importance.csv | Time: 0:16:51.190249
Processing model_1 | Type: rf
Process: 100/1000
Process: 200/1000
Process: 300/1000
Process: 400/1000
Process: 500/1000
Process: 600/1000
Process: 700/1000
Process: 800/1000
Process: 900/1000
Process: 1000/1000
Save: results/model_1_rf_feature_importance.csv | Time: 0:17:54.098887
Processing model_2 | Type: xgboost
Process: 100/1000
Process: 200/1000
Process: 300/1000
Process: 400/1000
Process: 500/1000
Process: 600/1000
Process: 700/1000
Process: 800/1000
Process: 900/1000
Process: 1000/1000
Save: results/model_2_xgboost_feature_importance.csv | Time: 0:25:41.965831
Processing model_2 | Type: rf
Process: 100/1000
Process: 200/1000
Process: 300/1000
Process: 400/1000
Process: 500/1000
Process: 600/1